# Setup — Entregable 2: benchmark en Colab Pro

Este notebook deja la sesión **lista para entrenar** en 6 celdas:

1. Verificar GPU  
2. Instalar dependencias  
3. Montar Drive y fijar rutas  
4. Sincronizar datos a disco local (splits remapeados a rutas Linux)  
5. Verificar entorno + splits  
6. Dry-run del driver  

**Runtime → Change runtime type:**

| GPU | Coste CU/h | Recomendación |
|---|---|---|
| T4 | 1,19 | ✔ adecuada, la más económica |
| L4 | 1,71 | ✔ mejor relación velocidad/CU para este benchmark |
| A100 | 5,40 | ✘ **no asignar**: quema los ~100 CU de Pro 3–5× más rápido sin beneficio |

Los **resultados se escriben en Drive**, así que una desconexión pierde como
mucho la corrida en curso: basta reejecutar la última celda para reanudar.

> **Nota de rutas:** los CSV de `datos/splits/` contienen rutas absolutas
> Windows (`C:\Users\...`). La celda 4 las reescribe a rutas de Colab
> (`/content/...`) en una **copia local**, sin tocar Drive.

In [ ]:
# Celda 1 — GPU disponible
!nvidia-smi | head -15
import os
assert 'CUDA_VISIBLE_DEVICES' in os.environ or True  # informativo
print("\nSi 'CUDA' no aparece arriba: Runtime → Change runtime type → GPU (T4 o L4).")

In [ ]:
# Celda 2 — dependencias del benchmark
# torch/torchvision ya vienen preinstalados en Colab (versión CUDA correcta)
%pip install -q segmentation-models-pytorch monai einops timm scipy scikit-image pandas opencv-python-headless tqdm nnunetv2

# nnunetv2 va incluido: nnU-Net es uno de los 10 modelos del benchmark
# (se entrena como pipeline propio, ver modelos_tb.py y la Fase 2).

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")

In [ ]:
# Celda 3 — montar Drive y fijar rutas
from google.colab import drive
drive.mount('/content/drive')

# >>> AJUSTAR SI LA CARPETA DEL PROYECTO TIENE OTRO NOMBRE EN DRIVE <<<
# (carpeta compartida: https://drive.google.com/drive/folders/1wKDUCv4qxneLvUjhuAHFzI1oBHgn7_rv)
CARPETA_DRIVE = "/content/drive/MyDrive/proyecto_tb"
RAIZ_LOCAL = "/content/proyecto_tb"          # copia de trabajo en disco local
RESULTADOS = f"{CARPETA_DRIVE}/resultados"    # en Drive: sobreviven a la sesión

assert os.path.isdir(CARPETA_DRIVE), (
    f"No encuentro el proyecto en {CARPETA_DRIVE}.\n"
    "Monta Drive, navega a la carpeta y corrige CARPETA_DRIVE con la ruta real."
)
assert os.path.isdir(f"{CARPETA_DRIVE}/datos/splits"), "Falta datos/splits en Drive"
print("Drive OK :", CARPETA_DRIVE)
print("Local    :", RAIZ_LOCAL)
print("Resultados:", RESULTADOS)

In [ ]:
# Celda 4 — sincronizar datos a disco local y remapear rutas
# Copia SOLO lo necesario (splits + NPZ train/val + PNG del test) ~85 MB.
# Ventaja: las lecturas NPZ en disco local son ~10x más rápidas que sobre
# Google Drive FUSE, y los splits se reescriben con rutas Linux.
import csv, shutil, zipfile

# 4.0 si subiste la actualización del proyecto como zip, extraerla en Drive
zp = f"{CARPETA_DRIVE}/actualizacion_20261020.zip"
if os.path.exists(zp):
    with zipfile.ZipFile(zp) as z:
        z.extractall(CARPETA_DRIVE)
    print("actualización 2026-10-20 extraída en Drive")

def a_local(ruta_windows, raiz):
    r"""C:\Users\...\proyecto_tb\datos\x.npz -> {raiz}/datos/x.npz"""
    norm = ruta_windows.replace("\\", "/")
    i = norm.find("datos/")
    assert i >= 0, f"ruta sin prefijo datos/: {ruta_windows}"
    return os.path.join(raiz, norm[i:])

# 4.1 código (scripts del benchmark) — SIEMPRE refrescar desde Drive
shutil.copytree(f"{CARPETA_DRIVE}/codigo", f"{RAIZ_LOCAL}/codigo",
                dirs_exist_ok=True)
print("código copiado (actualizado desde Drive)")

# 4.2 splits: copiar CSVs con las rutas REESCRITAS a local + copiar datos
os.makedirs(f"{RAIZ_LOCAL}/datos/splits", exist_ok=True)
n_nuevos = 0
for nombre in ("train", "val", "test_pos", "test_neg", "test_pos_reserva"):
    origen = f"{CARPETA_DRIVE}/datos/splits/{nombre}.csv"
    with open(origen, newline="", encoding="utf-8") as f:
        filas = list(csv.DictReader(f))
    campos = list(filas[0].keys())
    for fila in filas:
        dst = a_local(fila["ruta"], RAIZ_LOCAL)
        src = a_local(fila["ruta"], CARPETA_DRIVE)
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
            n_nuevos += 1
        fila["ruta"] = dst
    with open(f"{RAIZ_LOCAL}/datos/splits/{nombre}.csv", "w",
              newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campos)
        w.writeheader()
        w.writerows(filas)
    print(f"  split {nombre:16s} -> {len(filas):3d} filas (rutas remapeadas)")

# 4.3 manifiestos sellados (v2 = línea base vigente; v1 solo histórico)
for mf in ("split_manifest_v2.json", "split_manifest.json"):
    origen = f"{CARPETA_DRIVE}/datos/splits/{mf}"
    if os.path.exists(origen):
        shutil.copy2(origen, f"{RAIZ_LOCAL}/datos/splits/{mf}")
assert os.path.exists(f"{RAIZ_LOCAL}/datos/splits/split_manifest_v2.json"), \
    "Falta split_manifest_v2.json (línea base de integridad)"

# 4.4 GT plata D1-R1 (generado y validado el 2026-10-20; OBLIGATORIO)
gt_drive = f"{CARPETA_DRIVE}/datos/ground_truth_test"
if os.path.isdir(gt_drive):
    shutil.copytree(gt_drive, f"{RAIZ_LOCAL}/datos/ground_truth_test",
                    dirs_exist_ok=True)
    print("  ground_truth_test (GT plata D1-R1) copiado")
else:
    raise FileNotFoundError(
        "Falta ground_truth_test en Drive: sube el GT plata validado "
        "(80 PNG + ANOTACION_METRICAS.json) antes de cualquier corrida.")

print(f"Archivos de datos copiados esta sesión: {n_nuevos}")

In [ ]:
# Celda 5 — diagnóstico de modelos + verificación de splits
import sys
sys.path.insert(0, f"{RAIZ_LOCAL}/codigo")

from modelos_tb import resumen_entorno
from dataset_tb import verificar_splits

resumen_entorno()
print()
# esperar_gt=True: el GT plata D1-R1 ya está congelado y validado (§9.5);
# si falla aquí, es que Drive no tiene ground_truth_test/ completo.
verificar_splits(splits_dir=f"{RAIZ_LOCAL}/datos/splits", esperar_gt=True)

In [ ]:
# Celda 6 — dry-run: el driver solo planifica, NO entrena nada.
# Debe listar ~50 corridas pendientes y el diagnóstico de modelos.
!python "{RAIZ_LOCAL}/codigo/driver_entrenamiento.py" --dry-run --splits-dir "{RAIZ_LOCAL}/datos/splits" --resultados "{RESULTADOS}"

## Entrenar (fuera de este setup)

El driver es **reanudable**: escribe `resultado.json` por corrida en Drive.
Si la sesión cae, reejecutar el **mismo comando** retoma donde quedó
(solo se pierde la corrida en curso).

```python
# Celda de entrenamiento — bloque sugerido (corta limpio a los 5,5 h y guarda)
!python "{RAIZ_LOCAL}/codigo/driver_entrenamiento.py" \
    --splits-dir "{RAIZ_LOCAL}/datos/splits" \
    --resultados "{RESULTADOS}" \
    --max-minutos 330
```

### Protocolo de sesión

| Regla | Detalle |
|---|---|
| Corte por sesión | `--max-minutos 330` (5,5 h): guarda y sale limpio antes del tope de ~12 h de sesión Pro |
| Inactividad | irrelevante mientras entrena; el peligro es el código **detenido** >90 min — no pausar la celda |
 | Presupuesto CU | L4 = 1,71 CU/h → 5,5 h ≈ 9,4 CU por bloque; ~100 CU de Pro ≈ 10–12 bloques de 5,5 h |
| A100 | no usar (5,40 CU/h) |
| Reanudar | reejecutar la misma celda; el plan se recalcula solo |
| Progreso | celda inferior: `resumen_runs.csv` |

### Orden sugerido

1. **Semana 1:** `--modelos unet attention_unet unet_pp` (CNN, baratas) → validar el pipeline completo de punta a punta.
2. **Semana 2:** `--modelos deeplabv3p segformer transunet swin_unet` (mixtos).
3. **Semana 3:** `--modelos unetr mednext nnunet` (las más pesadas) + `unet_fuerte`.
4. **El GT plata ya está congelado y validado** (D1-R1, §9.5): no modificar `ground_truth_test/`; evaluar directamente.

In [ ]:
# Celda de ENTRENAMIENTO — esta es la celda que entrena (la 6 solo hace dry-run).
# Reanudable: si la sesión cae, reejecuta esta misma celda y continúa.
# Sin --modelos corre los 10 modelos + baselines (50 corridas en total);
# para validar primero el pipeline con la más barata, agrega:
#     --modelos unet --max-minutos 120
!python "{RAIZ_LOCAL}/codigo/driver_entrenamiento.py" \
    --splits-dir "{RAIZ_LOCAL}/datos/splits" \
    --resultados "{RESULTADOS}" \
    --max-minutos 330

In [ ]:
# Celda de ENTRENAMIENTO nnU-Net — los 5 folds (reanumable).
# Equivalencia folds<->semillas: 0->42, 1->123, 2->777, 3->2024, 4->31415.
# 1) convertir PNG/npz -> NIfTI y planificar (~1-2 min, una vez por sesion)
!python "{RAIZ_LOCAL}/codigo/entrenamiento_nnunet.py" --solo-preparar --resultados "{RESULTADOS}"

# 2) entrenar y evaluar los 5 folds (lo mas largo del benchmark; si la
#    sesion cae, re-ejecuta ESTA misma celda: los folds con resultado.json
#    se omiten automaticamente)
!python "{RAIZ_LOCAL}/codigo/entrenamiento_nnunet.py" --resultados "{RESULTADOS}"

# Al terminar: resultado.json por fold en resultados/nnunet/seed_<semilla>/
# y resumen_runs.csv regenerado incluyendo nnunet.

In [ ]:
# Celda 7 — progreso del benchmark (ejecutar cuando quieras ver el estado)
import pandas as pd
ruta = f"{RESULTADOS}/resumen_runs.csv"
if os.path.exists(ruta):
    df = pd.read_csv(ruta)
    print(f"{len(df)} corridas registradas | "
          f"completadas: {(df.estado == 'completado').sum()}")
    display(df.sort_values(["modelo", "semilla"]))
else:
    print("Aún no hay corridas. Lanza la celda de entrenamiento.")